# Adaptive Static Malware Detection — SageMaker training and system validation

**AAI-540 · Group 1 · branch `marco-model-training-evaluation`**

Run this notebook from a cloned repository in SageMaker Studio/JupyterLab or a
notebook instance with Python 3.12. Each teammate supplies an existing private S3
bucket, region, and SageMaker execution role in their own account. This notebook
contains no credentials and does not assume access to another member's account.
All AWS writes and paid jobs require the switches below. Running the unedited
notebook does not start a training job or create a hosted endpoint.

Read [the runbook](../docs/sagemaker_runbook.md), [system design](../docs/ml_system_design.md),
[rubric evidence map](../docs/rubric_coverage.md), and [video outline](../docs/demo_outline.md).
Save evidence from your actual run; the repository contains no invented AWS results.
The newest reserved-production period remains untouched throughout this notebook.

## 1. Install and locate the repository

Run the next cell once in a fresh kernel, then **restart the kernel** if an older
SageMaker SDK was already imported. The implementation uses SDK v2 and explicit
container versions. Do not upgrade to SDK v3 without migrating the API calls.
Installation downloads software but creates no AWS resources.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "malware_mlops").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Clone this branch and open the notebook from that checkout.")
os.chdir(ROOT)
INSTALL_DEPENDENCIES = False  # Set True once if requirements are not installed.
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements-sagemaker.txt"])
for directory in [ROOT / "src", ROOT / "sagemaker_src"]:
    sys.path.insert(0, str(directory))
print("Repository:", ROOT)

## 2. Configure your account and select the actions to run

Copy `configs/sagemaker.example.json` to the ignored `configs/sagemaker.local.json`.
Edit `member`, `aws.region`, `aws.bucket`, `aws.role_arn`, and `data.handoff_s3_uri`.
Use temporary Learner Lab credentials or the notebook execution role through the
normal AWS credential provider chain. Do not paste keys into notebook cells.

The handoff must include `feature_schema.json`, `temporal_splits_v1/split_manifest.json`,
and train/validation/test CSVs in the README layout. Earlier schemas fitted scaling
and imputation on 52 official train weeks, including validation. Regenerate those
artifacts with this branch's `prepare_features.py` and `create_temporal_splits.py`
before strict training. `allow_legacy_preprocessing=true` is an explicit reproduction
exception whose metrics must disclose that limitation; it does not repair the data.

Targets in `quality` are proposed acceptance criteria, not measured results. Agree
on them before examining test results. Failed test gates block registration; do not
retune against the same test set. Test is an **official-test-derived temporal subset**.

In [ ]:
import json
from malware_mlops.aws_workflow import (
    load_config, create_aws_context, download_handoff, validate_handoff,
    upload_handoff, ensure_feature_group, build_pipeline, run_pipeline,
    export_execution_report, get_registered_package, collect_evaluation,
    run_batch_transform, publish_observability, cleanup_owned_resources,
)

RUN_AWS = False
DOWNLOAD_HANDOFF = False
RUN_LOCAL_TRAINING = False
RUN_FEATURE_STORE = False
RUN_FAILURE_DEMO = False
RUN_BATCH = False
APPROVE_MODEL = False
PUBLISH_DASHBOARD = False
RUN_CLEANUP = False

config_path = Path(os.environ.get("AAI540_CONFIG", "configs/sagemaker.local.json"))
if not config_path.exists():
    config_path = Path("configs/sagemaker.example.json")
overrides = {"aws": {}, "data": {}}
for env_name, section, key in [
    ("AWS_REGION", "aws", "region"), ("AAI540_BUCKET", "aws", "bucket"),
    ("SAGEMAKER_ROLE_ARN", "aws", "role_arn"),
    ("AAI540_HANDOFF_S3_URI", "data", "handoff_s3_uri"),
]:
    if os.environ.get(env_name):
        overrides[section][key] = os.environ[env_name]
if os.environ.get("AAI540_MEMBER"):
    overrides["member"] = os.environ["AAI540_MEMBER"]
cfg = load_config(config_path, overrides)
cfg["cloud_execution"] = RUN_AWS
REPORT_DIR = ROOT / cfg["reports_dir"] / cfg["run_id"]
REPORT_DIR.mkdir(parents=True, exist_ok=True)
context = None
handoff = None
execution = None
evidence = None
evaluation_report = None
model_package_arn = None
local_model_dir = None
inventory = {}
print(json.dumps({"member": cfg["member"], "run_id": cfg["run_id"],
                  "region": cfg["aws"]["region"], "cloud_execution": RUN_AWS,
                  "quality": cfg["quality"]}, indent=2))

## 3. Architecture and operational boundaries

The S3 handoff is the immutable full training snapshot. Feature Store demonstrates
governed, reusable features using a small training-only sample; it is not silently
substituted for that full snapshot. The registry retains candidate versions pending
manual approval. Batch Transform provides inference without an always-on endpoint.
The monitoring reports below are custom PSI and labeled-performance checks;
this implementation does not create a SageMaker Model Monitor schedule.

In [ ]:
from IPython.display import SVG, display
display(SVG('''<svg xmlns="http://www.w3.org/2000/svg" width="1000" height="250" viewBox="0 0 1000 250">
<style>text{font:14px sans-serif;fill:#162b45}rect{fill:#eff5ff;stroke:#3763a2;rx:8}path{stroke:#3763a2;fill:none;stroke-width:2}</style>
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6"/></marker></defs>
<rect x="10" y="30" width="160" height="65"/><text x="25" y="56">S3 handoff + schema</text><text x="25" y="79">Train / validation / test</text>
<rect x="220" y="30" width="170" height="65"/><text x="235" y="56">Managed training</text><text x="235" y="79">Validation selection</text>
<rect x="440" y="30" width="160" height="65"/><text x="455" y="56">Test evaluation</text><text x="455" y="79">Quality gate / Fail</text>
<rect x="650" y="30" width="150" height="65"/><text x="665" y="56">Model Registry</text><text x="665" y="79">Manual approval</text>
<rect x="850" y="30" width="140" height="65"/><text x="865" y="56">Batch Transform</text><text x="865" y="79">S3 predictions</text>
<path d="M170,62 H218 M390,62 H438 M600,62 H648 M800,62 H848" marker-end="url(#arrow)"/>
<rect x="15" y="155" width="230" height="60"/><text x="30" y="180">Feature Store</text><text x="30" y="202">Training sample + online readback</text>
<rect x="345" y="155" width="300" height="60"/><text x="360" y="180">CloudWatch + custom monitoring</text><text x="360" y="202">Infrastructure / drift / delayed labels</text>
<rect x="745" y="155" width="240" height="60"/><text x="760" y="180">GitHub CI → optional AWS job</text><text x="760" y="202">Tests / notebooks / OIDC</text>
<path d="M90,95 V153 M520,95 V153" marker-end="url(#arrow)"/>
</svg>'''))

## 4. Account preflight and S3 handoff

With `RUN_AWS=True`, confirm the identity, bucket region, and role resolved by the
preflight. A Learner Lab account may deny Feature Store, Pipelines, registration,
CloudWatch writes, `iam:PassRole`, or a requested instance type. Resolve the specific
permission/quota with the account owner and record the limitation; do not label a
skipped service as a successful demonstration. The code never creates an IAM role.
`DOWNLOAD_HANDOFF` downloads only named train/validation/test artifacts, never the reserve.

In [ ]:
if RUN_AWS:
    context = create_aws_context(cfg)
    print(json.dumps({"identity": context["identity"], "region": context["region"],
                      "role_arn": context["role_arn"]}, indent=2))
    if DOWNLOAD_HANDOFF:
        downloaded = download_handoff(cfg, context)
        print("Downloaded:", sorted(downloaded))
else:
    print("AWS actions disabled. Configure the local JSON and switches to run in your account.")

processed_dir = ROOT / cfg["data"]["processed_dir"]
if (processed_dir / "feature_schema.json").exists():
    handoff = validate_handoff(cfg)
    print(json.dumps({"schema_sha256": handoff["schema_sha256"], "splits": handoff["splits"],
                      "reserved_production": handoff["reserved_production"]}, indent=2))
else:
    print("No local handoff yet. Request the regenerated S3 artifacts from the data owner.")

## 5. Inspect the data and feature contract

The validator checks ordered numeric features, binary labels, unique IDs, manifest
row counts and schema fingerprint, and chronological separation. Identifiers and
lineage never enter `X`. Charts load only the three selected splits; the production
reserve is not opened. The model uses 13 compact static PE aggregates, not raw malware
binaries, hashes, antivirus scores, timestamps, or malware-family labels.

In [ ]:
if handoff is not None:
    import pandas as pd
    import matplotlib.pyplot as plt
    columns = handoff["schema"]["model_feature_columns"]
    counts = pd.DataFrame({name: details["class_distribution"]
                           for name, details in handoff["splits"].items()}).T
    counts = counts.rename(columns={"0": "Benign", "1": "Malicious"}).astype(int)
    counts.plot.bar(stacked=True, title="Chronological split class counts", figsize=(8, 4))
    plt.ylabel("Unique records")
    plt.tight_layout()
    plt.show()
    print("Feature order:", columns)
    print("Preprocessing scope:", handoff["schema"].get("fit_scope"),
          "fit_end_week:", handoff["schema"].get("fit_end_week"))
else:
    print("Data inspection waits for the handoff.")

## 6. Feature Store and feature group demonstration

Enable both `RUN_FEATURE_STORE` here and `feature_store.enabled` in the local config.
The function creates a group owned by this run, ingests a bounded sample of training
rows, waits for group readiness, and verifies an online `GetRecord`. Offline S3
delivery is eventually consistent. Record the group description, feature definitions,
online readback, and offline URI for the video. No validation/test rows are ingested.

In [ ]:
if RUN_AWS and RUN_FEATURE_STORE:
    if handoff is None:
        raise RuntimeError("Validate the handoff first.")
    feature_result = ensure_feature_group(cfg, context, handoff)
    inventory.update(feature_result.get("inventory", {}))
    (REPORT_DIR / "feature_store_evidence.json").write_text(json.dumps(feature_result, indent=2, default=str))
    display(feature_result)
else:
    print("Feature Store disabled; this criterion needs an actual account run.")

## 7. Optional local training check

This path uses the same managed entry points and your downloaded data without
creating a SageMaker job. It is useful for debugging before spending AWS credits.
Fixed candidates are logistic regression, random forest, and histogram gradient
boosting. Selection maximizes validation recall subject to `max_fpr`, then breaks
ties using FPR and PR AUC. Threshold ties are handled as complete score groups.
Training defaults to a deterministic stratified cap of 250,000 training rows for
CPU cost; validation and test stay complete. Use `--max-train-rows 0` for all training rows.
This section is optional; the managed DAG is the required AWS demonstration.

In [ ]:
if RUN_LOCAL_TRAINING:
    if handoff is None:
        raise RuntimeError("Validate the handoff first.")
    local_model_dir = REPORT_DIR / "local" / "model"
    local_eval_dir = REPORT_DIR / "local" / "evaluation"
    subprocess.check_call([
        sys.executable, "sagemaker_src/train.py", "--train-dir", str(Path(handoff["paths"]["train"]).parent),
        "--validation-dir", str(Path(handoff["paths"]["validation"]).parent),
        "--schema-dir", str(Path(handoff["paths"]["schema"]).parent),
        "--model-dir", str(local_model_dir), "--output-dir", str(REPORT_DIR / "local" / "training"),
        "--max-fpr", str(cfg["quality"]["max_fpr"]), "--seed", str(cfg["compute"]["seed"]),
        "--allow-legacy-preprocessing", str(cfg["data"]["allow_legacy_preprocessing"]).lower(),
    ])
    subprocess.check_call([
        sys.executable, "sagemaker_src/evaluate.py", "--model-dir", str(local_model_dir),
        "--test-dir", str(Path(handoff["paths"]["test"]).parent),
        "--schema-dir", str(Path(handoff["paths"]["schema"]).parent), "--output-dir", str(local_eval_dir),
    ])
    evaluation_report = json.loads((local_eval_dir / "evaluation.json").read_text())
    print("Local check completed. AWS operation remains unvalidated by a local run.")
else:
    print("Local training disabled.")

## 8. Managed SageMaker Pipeline: train, evaluate, gate, register

With `RUN_AWS=True`, this cell uploads validated inputs, creates/updates this run's
pipeline, and starts paid training and processing jobs. Training receives only train,
validation, and schema channels. Evaluation receives the frozen model and test.
Only a passing gate registers a version as `PendingManualApproval`. The generated
pipeline definition is saved beside execution evidence for review.

Wait for the run to finish before downstream cells. A failed quality gate is useful
evidence of a blocked candidate; changing criteria after looking at test scores
would invalidate the holdout protocol. Job runtimes and instance availability vary.

In [ ]:
if RUN_AWS:
    if handoff is None:
        raise RuntimeError("Validate the regenerated handoff first.")
    inputs = upload_handoff(cfg, context, handoff)
    pipeline = build_pipeline(cfg, context, inputs)
    (REPORT_DIR / "pipeline_definition.json").write_text(pipeline.definition(), encoding="utf-8")
    execution = run_pipeline(cfg, context, inputs, force_failure=False, wait=False)
    inventory["pipeline_name"] = pipeline.name
    print("Execution:", execution.arn)
    print("Use SageMaker Studio's pipeline graph for the recorded DAG demonstration.")
else:
    print("Managed training disabled. Set RUN_AWS only after reviewing account configuration.")

In [ ]:
if execution is not None:
    try:
        execution.wait(delay=30, max_attempts=360)
    except Exception as error:
        print("Managed run needs inspection:", type(error).__name__, str(error))
    finally:
        execution_report = export_execution_report(execution, REPORT_DIR)
        display(execution_report)
    if any(step.get("StepName") == "EvaluateTemporalTest" and step.get("StepStatus") == "Succeeded"
           for step in execution_report["steps"]):
        evidence = collect_evaluation(cfg, context, execution)
        evaluation_report = evidence["report"]
    if execution_report["execution"]["PipelineExecutionStatus"] == "Succeeded":
        model_package_arn = get_registered_package(execution)
        print("Registered package:", model_package_arn)
    else:
        print("No successful release claimed. Inspect failed/running steps and the saved report.")
else:
    print("No managed execution in this session.")

## 9. Review measured results and plots

Interpret recall together with FPR, precision, PR AUC, and the benign sample count.
An empirical low FPR is not a bound on production FPR, especially under temporal
drift and changing class prevalence. Weekly metrics reveal changes obscured by an
aggregate score. No results are available until a local or AWS run supplies them.

In [ ]:
if evaluation_report is not None:
    import pandas as pd
    import matplotlib.pyplot as plt
    if evidence and evidence.get("model_artifact_s3_uri") and local_model_dir is None:
        from evaluate import extract_model
        from malware_mlops.data import parse_s3_uri
        archive_dir = REPORT_DIR / "model_artifact"
        archive_dir.mkdir(exist_ok=True)
        bucket, key = parse_s3_uri(evidence["model_artifact_s3_uri"])
        context["s3"].download_file(bucket, key, str(archive_dir / "model.tar.gz"))
        local_model_dir = extract_model(archive_dir)
    if local_model_dir is not None:
        training_report = json.loads((local_model_dir / "training_report.json").read_text())
        comparison = pd.DataFrame([{ "candidate": row["name"], "recall": row["validation"]["recall"],
                                     "fpr": row["validation"]["fpr"], "pr_auc": row["validation"]["pr_auc"] }
                                    for row in training_report["candidates"]]).set_index("candidate")
        display(comparison)
        comparison[["recall", "pr_auc"]].plot.bar(title="Validation-only candidate comparison", figsize=(8, 4))
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.show()
        print("Selected:", training_report["selected_model"], "training rows:", training_report["train"]["rows"])
    display(pd.DataFrame({name: value["value"] for name, value in
                          evaluation_report["classification_metrics"].items()}, index=["held_out_test"]).T)
    print("Frozen threshold:", evaluation_report["threshold"])
    print("Confusion matrix:", evaluation_report["confusion_matrix"])
    weekly = pd.DataFrame(evaluation_report.get("weekly_metrics", {})).T
    if not weekly.empty:
        weekly[["recall", "fpr"]].plot(marker="o", figsize=(9, 4), title="Weekly held-out test performance")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()
    plot_dir = Path(evidence["directory"]) if evidence else REPORT_DIR / "local" / "evaluation"
    if not list(plot_dir.glob("*.png")) and local_model_dir is not None:
        # The managed sklearn image may omit matplotlib. Plot the same frozen
        # model locally; this does not change selection, threshold, or test metrics.
        import joblib
        from training import evaluate_bundle, load_frame
        from evaluate import save_plots
        test_frame = load_frame(Path(handoff["paths"]["test"]), handoff["schema"])
        frozen_bundle = joblib.load(local_model_dir / "model.joblib")
        _, frozen_probabilities = evaluate_bundle(frozen_bundle, test_frame, handoff["schema"])
        save_plots(test_frame, frozen_probabilities, frozen_bundle["threshold"], plot_dir)
        print("Figures rendered in the notebook kernel from the frozen model and unchanged test data.")
    from IPython.display import Image
    for path in sorted(plot_dir.glob("*.png")):
        display(Image(filename=str(path)))
else:
    print("No measured scores yet. Run training and evaluation to populate this section.")

## 10. Demonstrate the failed DAG without changing scientific criteria

`RUN_FAILURE_DEMO=True` starts another managed execution with `ForceFail=true`.
It reaches an explicit failure branch and cannot register a model. This may incur
another training/processing charge because job caching is not assumed. Capture its
failed graph alongside a real passing execution. A forced demonstration failure is
not a genuine model-quality failure; label it accurately in the video. GitHub CI is
separate and must also show actual green/red checks if your instructor expects both.

In [ ]:
if RUN_AWS and RUN_FAILURE_DEMO:
    failed_execution = run_pipeline(cfg, context, inputs, force_failure=True, wait=False)
    try:
        failed_execution.wait()
    except Exception as error:
        print("Inspect the recorded failed step and reason:", type(error).__name__, str(error))
    finally:
        failed_report = export_execution_report(failed_execution, REPORT_DIR / "failure_demo")
        display(failed_report)
    rejected = any(step.get("StepName") == "RejectCandidate" and step.get("StepStatus") == "Failed"
                   for step in failed_report["steps"])
    if failed_report["execution"]["PipelineExecutionStatus"] != "Failed" or not rejected:
        raise RuntimeError("Expected RejectCandidate failure; an infrastructure or permission failure does not satisfy this demonstration.")
else:
    print("Forced failure demonstration disabled.")

## 11. Model Registry review and batch inference

Review the registered version, evaluation report, preprocessing scope, and artifact
lineage. `APPROVE_MODEL=True` records your explicit reviewer decision in your own
account; it is independent of the automated gate. `RUN_BATCH=True` then serves that
approved package with Batch Transform. The input contains only schema-ordered features,
and the output is `probability,label` per row. A separate mapping retains record IDs.
The demonstration uses a test sample and does not open reserved production.

In [ ]:
if RUN_AWS and model_package_arn:
    package = context["sagemaker"].describe_model_package(ModelPackageName=model_package_arn)
    print("Package:", model_package_arn, "approval:", package["ModelApprovalStatus"])
    if APPROVE_MODEL:
        context["sagemaker"].update_model_package(
            ModelPackageArn=model_package_arn, ModelApprovalStatus="Approved",
            ApprovalDescription="Reviewed quality report, schema, and validation-only selection for this demonstration.",
        )
        print("Reviewer approval recorded.")
    if RUN_BATCH:
        import pandas as pd
        sample = pd.read_csv(handoff["paths"]["test"], nrows=100)
        batch_path = REPORT_DIR / "batch_features.csv"
        sample[handoff["schema"]["model_feature_columns"]].to_csv(batch_path, header=False, index=False)
        sample[["record_id"]].to_csv(REPORT_DIR / "batch_record_mapping.csv", index_label="row_number")
        batch_result = run_batch_transform(cfg, context, model_package_arn, batch_path)
        inventory.update(batch_result.get("inventory", {}))
        if evidence is not None:
            evidence["job_names"]["batch"] = batch_result["transform_job_name"]
        (REPORT_DIR / "batch_evidence.json").write_text(json.dumps(batch_result, indent=2, default=str))
        display(batch_result)
        display(pd.read_csv(batch_result["local_predictions"], header=None,
                            names=["probability", "predicted_label"]).head(10))
elif RUN_BATCH or APPROVE_MODEL:
    raise RuntimeError("A real passing execution and registered package are required first.")
else:
    print("Registry review and batch actions wait for a passing managed execution.")

## 12. Data/model monitoring and infrastructure dashboard

Custom monitoring compares incoming feature histograms to the training reference,
reports PSI, missingness, and training-range violations, and optionally evaluates
delayed labels with the already frozen threshold. PSI is a screening heuristic;
distribution shift is not proof of degraded classification. Label-based performance
is needed to assess concept drift. The following example uses test as a historical
newer-time monitoring window, not a live production feed.

For a managed run, download only the registered training artifact from the returned
S3 URI and extract its allowlisted files using `evaluate.extract_model`. Never load
an untrusted pickle/joblib artifact. CloudWatch publishes run-level metrics and a
dashboard containing infrastructure and custom model/data quality views. Set alarms
and notification ownership for a deployed operational service; this notebook creates
no automatic retraining or alert subscription.

In [ ]:
drift_report = None
if evaluation_report is not None and handoff is not None:
    from evaluate import extract_model
    from malware_mlops.data import parse_s3_uri
    if evidence and evidence.get("model_artifact_s3_uri") and local_model_dir is None:
        archive_dir = REPORT_DIR / "model_artifact"
        archive_dir.mkdir(exist_ok=True)
        bucket, key = parse_s3_uri(evidence["model_artifact_s3_uri"])
        context["s3"].download_file(bucket, key, str(archive_dir / "model.tar.gz"))
        local_model_dir = extract_model(archive_dir)
    if local_model_dir is not None:
        subprocess.check_call([
            sys.executable, "sagemaker_src/monitor.py",
            "--reference-path", str(local_model_dir / "reference_histograms.json"),
            "--data-path", handoff["paths"]["test"], "--schema-path", handoff["paths"]["schema"],
            "--model-dir", str(local_model_dir), "--output-dir", str(REPORT_DIR / "monitoring"),
        ])
        drift_report = json.loads((REPORT_DIR / "monitoring" / "monitoring_report.json").read_text())
        display(drift_report)
        from IPython.display import Image
        for path in (REPORT_DIR / "monitoring").glob("*.png"):
            display(Image(filename=str(path)))

if RUN_AWS and PUBLISH_DASHBOARD:
    if evaluation_report is None:
        raise RuntimeError("Measured evaluation results are required before publishing metrics.")
    dashboard_result = publish_observability(
        cfg, context, evaluation_report,
        job_names=evidence.get("job_names", {}) if evidence else {}, drift_report=drift_report,
    )
    inventory.update(dashboard_result.get("inventory", {}))
    (REPORT_DIR / "dashboard_evidence.json").write_text(json.dumps(dashboard_result, indent=2, default=str))
    display(dashboard_result)
else:
    print("Dashboard publishing disabled.")

## 13. Evidence, CI/CD, and cleanup

Run `python -m unittest discover -s tests -v` and
`python scripts/validate_notebooks.py` locally or inspect `.github/workflows/ci.yml`.
The optional AWS workflow requires explicitly configured GitHub OIDC trust and account
variables. These checks do not by themselves validate a deployed AWS system.

Collect the Feature Store group/readback, successful and failed pipeline DAGs, model
package ARN/status, batch S3 output and sample predictions, monitoring JSON/charts,
and populated CloudWatch dashboard. Record equal speaking portions using the video
outline. The video, peer evaluation, submission delegate, and final stakeholder PDF
are team deliverables. Do not claim they were completed by running this notebook.

Training/processing/transform jobs terminate. Feature Store and retained S3 objects
can continue to incur charges. Review the inventory before explicit cleanup. Cleanup
removes only named resources owned by this run; it retains datasets, registry evidence,
and S3 outputs for review. Apply your account's retention policy separately.

In [ ]:
print("Evidence directory:", REPORT_DIR)
print("Run ID:", cfg["run_id"])
display(inventory)
if RUN_AWS and RUN_CLEANUP:
    cleanup_result = cleanup_owned_resources(cfg, context, inventory, confirm_run_id=cfg["run_id"])
    display(cleanup_result)
else:
    print("Cleanup disabled. Review the run inventory and set RUN_CLEANUP=True when ready.")

## References and interpretation

- [AWS SageMaker scikit-learn support](https://docs.aws.amazon.com/sagemaker/latest/dg/sklearn.html)
- [AWS container support policy](https://docs.aws.amazon.com/sagemaker/latest/dg/pre-built-containers-support-policy.html)
- [SageMaker Pipelines](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines.html)
- [Feature Store security](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store-security.html)
- [Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html)
- [Batch Transform](https://docs.aws.amazon.com/sagemaker/latest/dg/batch-transform.html)
- [CloudWatch SageMaker metrics](https://docs.aws.amazon.com/sagemaker/latest/dg/monitoring-cloudwatch.html)

EMBER2024 is a dated Win64 research sample. The compact feature set and empirical
acceptance gates do not establish adversarial robustness or authorize automatic file
deletion/blocking. Use human review, external temporal validation, and updated datasets
before operational security decisions. Model/artifact hashes and the saved code revision
support reproducibility; they do not establish provenance of an unverified dataset copy.